# AgroScan — Dataset fusionné PlantVillage + PlantWild → Fine-tuning PlantDoc

## Stratégie

Ce notebook implémente une nouvelle approche : au lieu d'entraîner séquentiellement sur PlantVillage puis PlantWild, on utilise un **dataset déjà fusionné** (PlantVillage + PlantWild) disponible sur Kaggle.

1. **Phase 1** — Entraînement sur le dataset fusionné (backbone gelé puis dégel partiel)
2. **Phase 2** — Fine-tuning léger sur PlantDoc (domaine cible)
3. **Évaluation comparative** — PlantVillage seul vs PlantDoc seul

```bash
./scripts/tf_gpu_env.sh uv run jupyter lab
```

## 1. Dépendances

In [1]:
# %pip install -r ../requirements.txt kagglehub

In [2]:
import os
import json
import shutil
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

import tensorflow as tf
import tensorflow_datasets as tfds

print('TensorFlow :', tf.__version__)
print('GPU        :', tf.config.list_physical_devices('GPU'))

2026-05-25 13:00:08.067698: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779714008.307965      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779714008.374386      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779714008.899504      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779714008.899546      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779714008.899549      57 computation_placer.cc:177] computation placer alr

TensorFlow : 2.19.0
GPU        : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


## 2. Configuration

In [3]:
from pathlib import Path

# On pointe vers le répertoire de travail de Kaggle
MODEL_DIR = Path('/kaggle/working/models')

# On s'assure que le dossier existe avant d'y sauvegarder des choses
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# --- Détection robuste de la racine projet ---
_env_root      = Path(os.environ['AGROSCAN_ROOT']) if 'AGROSCAN_ROOT' in os.environ else None
_colab_root    = Path('/content') if Path('/content').exists() and Path('/opt/google').exists() else None
_cwd           = Path('.').resolve()
try:
    _starting_dir = Path(get_ipython().starting_dir).resolve()
except (NameError, AttributeError):
    _starting_dir = _cwd

_candidates = list(dict.fromkeys(filter(None, [
    _env_root, _cwd, _starting_dir, *_cwd.parents, *_starting_dir.parents
])))
ROOT = next(
    (p for p in _candidates if (p / 'requirements.txt').exists()),
    _env_root or _colab_root or _cwd,
)

DATA_DIR   = ROOT / 'data'
# MODEL_DIR  = ROOT / 'models'
REPORT_DIR = ROOT / 'reports'

for d in [MODEL_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Hyperparamètres ---
IMG_SIZE       = (224, 224)
BATCH_SIZE     = 32
SEED           = 42
AUTOTUNE       = tf.data.AUTOTUNE

# Phase 1 : entraînement sur le dataset fusionné
LR_P1          = 1e-3    # backbone gelé
LR_P1_UNFREEZE = 1e-4    # dégel partiel
EPOCHS_P1_HEAD = 5       # head seule
EPOCHS_P1_FULL = 8       # avec dégel partiel
UNFREEZE_RATIO = 0.75    # proportion de couches gelées (les 75% premières restent gelées)

# Phase 2 : fine-tuning PlantDoc
LR_P2          = 5e-5
EPOCHS_P2      = 10

print('ROOT       :', ROOT)
print('DATA_DIR   :', DATA_DIR)
print('MODEL_DIR  :', MODEL_DIR)
print('REPORT_DIR :', REPORT_DIR)

ROOT       : /
DATA_DIR   : /data
MODEL_DIR  : /kaggle/working/models
REPORT_DIR : /reports


## 3. Téléchargement du dataset fusionné (PlantVillage + PlantWild)

Le dataset Kaggle `enasqtait/merged-dataset-plantvillage-and-plantwild` combine directement PlantVillage et PlantWild dans une structure `train/val/test` prête à l'emploi.

In [4]:
import kagglehub

print('Récupération du dataset via kagglehub...')
path_str = kagglehub.dataset_download("enasqtait/merged-dataset-plantvillage-and-plantwild")
dataset_path = Path(path_str)

# 1. Trouver automatiquement le dossier racine qui contient les classes
first_image = next(dataset_path.rglob('*.jpg'), None)
if first_image is None:
    first_image = next(dataset_path.rglob('*.png'), None) # Fallback pour les PNG

if first_image is None:
    raise FileNotFoundError("Aucune image trouvée dans le dataset !")

DATASET_ROOT = first_image.parent.parent
print(f"Racine du dataset trouvée : {DATASET_ROOT}")

# Petite vérification du nombre de classes
classes = [d.name for d in DATASET_ROOT.iterdir() if d.is_dir()]
print(f"Nombre de classes détectées : {len(classes)}")

# 2. Chargement et Split à la volée avec Keras
BATCH_SIZE = 32
IMG_SIZE = (224, 224) 
SEED = 123 

print("\n--- Création des datasets ---")

# Train Dataset (80%)
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_ROOT,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Validation Dataset (20%)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_ROOT,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# 3. Séparer le Validation set pour créer un Test set
val_batches = tf.data.experimental.cardinality(val_ds)
test_ds = val_ds.take(val_batches // 2)
val_ds = val_ds.skip(val_batches // 2)

print(f"Batches d'entraînement : {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"Batches de validation  : {tf.data.experimental.cardinality(val_ds).numpy()}")
print(f"Batches de test        : {tf.data.experimental.cardinality(test_ds).numpy()}")

Récupération du dataset via kagglehub...
Racine du dataset trouvée : /kaggle/input/datasets/enasqtait/merged-dataset-plantvillage-and-plantwild/final_dataset/kaggle/working/final_dataset
Nombre de classes détectées : 80

--- Création des datasets ---
Found 36103 files belonging to 80 classes.
Using 28883 files for training.


I0000 00:00:1779714047.100525      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779714047.106546      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 36103 files belonging to 80 classes.
Using 7220 files for validation.
Batches d'entraînement : 903
Batches de validation  : 113
Batches de test        : 113


In [5]:
merged_classes = sorted(p.name for p in DATASET_ROOT.iterdir() if p.is_dir())
print(f'Classes dans le dataset : {len(merged_classes)}')
print()

# Compter les images par classe (dataset global)
image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
class_counts = {}

for cls_dir in DATASET_ROOT.iterdir():
    if cls_dir.is_dir():
        # Compte les fichiers qui sont des images
        n = sum(1 for f in cls_dir.iterdir() if f.is_file() and f.suffix.lower() in image_exts)
        class_counts[cls_dir.name] = n

total_images = sum(class_counts.values())
print(f'Total images (global) : {total_images:,}')
print(f'Moyenne par classe    : {total_images / len(class_counts):.0f}')
print()

# Top 10 / Bottom 10
df_dist = pd.Series(class_counts).sort_values(ascending=False)
print('Top 10 classes :')
print(df_dist.head(10).to_string())
print('\nBottom 10 classes :')
print(df_dist.tail(10).to_string())

Classes dans le dataset : 80

Total images (global) : 36,103
Moyenne par classe    : 451

Top 10 classes :
Tomato__Tomato_YellowLeaf__Curl_Virus          3208
Tomato_Bacterial_spot                          2127
Tomato_Late_blight                             1909
Tomato_Septoria_leaf_spot                      1771
Pepper__bell___healthy                         1718
Tomato_Spider_mites_Two_spotted_spider_mite    1676
Tomato_healthy                                 1591
Tomato__Target_Spot                            1404
Potato___Late_blight                           1240
Potato___Early_blight                          1227

Bottom 10 classes :
grape leaf spot                     106
lettuce mosaic virus                100
strawberry anthracnose               98
cauliflower alternaria leaf spot     98
eggplant cercospora leaf spot        88
basil downy mildew                   86
strawberry leaf scorch               76
plum pocket disease                  76
carrot cavity spot              

## 4. Configuration PlantDoc (fine-tuning cible)

In [6]:
PLANTDOC_DIR       = DATA_DIR / 'plantdoc'
PLANTDOC_TRAIN_DIR = PLANTDOC_DIR / 'train'
PLANTDOC_TEST_DIR  = PLANTDOC_DIR / 'test'

# Mapping PlantDoc → noms du dataset fusionné (basé sur les noms PlantVillage standard)
plantdoc_to_merged = {
    'Apple Scab Leaf':                       'Apple___Apple_scab',
    'Apple leaf':                            'Apple___healthy',
    'Apple rust leaf':                       'Apple___Cedar_apple_rust',
    'Bell_pepper leaf':                      'Pepper,_bell___healthy',
    'Bell_pepper leaf spot':                 'Pepper,_bell___Bacterial_spot',
    'Blueberry leaf':                        'Blueberry___healthy',
    'Cherry leaf':                           'Cherry___healthy',
    'Corn Gray leaf spot':                   'Corn___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn leaf blight':                      'Corn___Northern_Leaf_Blight',
    'Corn rust leaf':                        'Corn___Common_rust',
    'Peach leaf':                            'Peach___healthy',
    'Potato leaf early blight':              'Potato___Early_blight',
    'Potato leaf late blight':               'Potato___Late_blight',
    'Raspberry leaf':                        'Raspberry___healthy',
    'Soyabean leaf':                         'Soybean___healthy',
    'Squash Powdery mildew leaf':            'Squash___Powdery_mildew',
    'Strawberry leaf':                       'Strawberry___healthy',
    'Tomato Early blight leaf':              'Tomato___Early_blight',
    'Tomato Septoria leaf spot':             'Tomato___Septoria_leaf_spot',
    'Tomato leaf':                           'Tomato___healthy',
    'Tomato leaf bacterial spot':            'Tomato___Bacterial_spot',
    'Tomato leaf late blight':               'Tomato___Late_blight',
    'Tomato leaf mosaic virus':              'Tomato___Tomato_mosaic_virus',
    'Tomato leaf yellow virus':              'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato mold leaf':                      'Tomato___Leaf_Mold',
    'Tomato two spotted spider mites leaf':  'Tomato___Spider_mites Two-spotted_spider_mite',
    'grape leaf':                            'Grape___healthy',
    'grape leaf black rot':                  'Grape___Black_rot',
}

if not PLANTDOC_TEST_DIR.exists():
    print('Téléchargement de PlantDoc...')
    PLANTDOC_DIR.mkdir(parents=True, exist_ok=True)
    tmp = DATA_DIR / '_pd_clone'
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/pratikkayal/PlantDoc-Dataset.git', str(tmp)],
        check=True,
    )
    for split in ('train', 'test'):
        src, dst = tmp / split, PLANTDOC_DIR / split
        if src.exists() and not dst.exists():
            shutil.copytree(src, dst)
    shutil.rmtree(tmp)

plantdoc_classes = sorted(p.name for p in PLANTDOC_TEST_DIR.iterdir() if p.is_dir())
print(f'PlantDoc — {len(plantdoc_classes)} classes dans test/')

Téléchargement de PlantDoc...


Cloning into '/data/_pd_clone'...
Updating files: 100% (2581/2581), done.


PlantDoc — 27 classes dans test/


## 5. Chargement PlantVillage (standalone, pour évaluation isolée)

On charge PlantVillage via `tensorflow_datasets` pour obtenir un test set propre, indépendant du dataset fusionné, afin de mesurer la rétention de performance sur les données studio.

In [7]:
(pv_test_raw,), pv_info = tfds.load(
    'plant_village',
    split=['train[85%:]'],
    as_supervised=True,
    with_info=True,
    data_dir=str(DATA_DIR / 'tfds'),
)

pv_class_names = pv_info.features['label'].names
pv_num_classes = len(pv_class_names)
pv_label_to_idx = {name: idx for idx, name in enumerate(pv_class_names)}

print(f'PlantVillage standalone — {pv_num_classes} classes')
print(f'Test set : {sum(1 for _ in pv_test_raw):,} images')

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /data/tfds/plant_village/incomplete.12I7HI_1.0.2/plant_village-train.tfrecord*...:   0%|          | …

Dataset plant_village downloaded and prepared to /data/tfds/plant_village/1.0.2. Subsequent calls will reuse this data.
PlantVillage standalone — 38 classes
Test set : 8,145 images


## 6. Construction des index de classes et mapping

Le dataset fusionné a ses propres noms de dossiers. On construit l'index depuis `MERGED_TRAIN_DIR` et on crée les correspondances nécessaires pour l'évaluation PlantVillage et PlantDoc.

In [8]:
# Index des classes du dataset fusionné (ordre alphabétique, identique à keras image_dataset_from_directory)
class_names   = sorted(p.name for p in DATASET_ROOT.iterdir() if p.is_dir())
num_classes   = len(class_names)
label_to_idx  = {name: idx for idx, name in enumerate(class_names)}

print(f'Dataset fusionné — {num_classes} classes')
print('Exemples :', class_names[:6])

# Mapping PlantVillage (tfds) → index du dataset fusionné
# Les noms PlantVillage tfds utilisent "Plant___Disease" ; le fusionné peut légèrement différer.
# On cherche d'abord une correspondance exacte, puis par inclusion.
pv_to_merged_idx = {}
for pv_cls in pv_class_names:
    if pv_cls in label_to_idx:
        pv_to_merged_idx[pv_cls] = label_to_idx[pv_cls]
    else:
        # Correspondance souple : cherche un nom fusionné qui contient le nom pv ou vice versa
        for m_cls in class_names:
            if pv_cls.lower() == m_cls.lower():
                pv_to_merged_idx[pv_cls] = label_to_idx[m_cls]
                break

print(f'\nClasses PlantVillage mappées vers le fusionné : {len(pv_to_merged_idx)} / {pv_num_classes}')
if pv_num_classes - len(pv_to_merged_idx) > 0:
    unmapped = [c for c in pv_class_names if c not in pv_to_merged_idx]
    print('Non mappées :', unmapped[:5], '...' if len(unmapped) > 5 else '')

# Mapping PlantDoc → index du dataset fusionné
valid_pd_mapping = {}
for pd_cls, merged_cls in plantdoc_to_merged.items():
    if pd_cls in plantdoc_classes and merged_cls in label_to_idx:
        valid_pd_mapping[pd_cls] = label_to_idx[merged_cls]

print(f'Classes PlantDoc évaluables : {len(valid_pd_mapping)} / {len(plantdoc_classes)}')

Dataset fusionné — 80 classes
Exemples : ['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot']

Classes PlantVillage mappées vers le fusionné : 10 / 38
Non mappées : ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy'] ...
Classes PlantDoc évaluables : 9 / 27


## 7. Augmentation et prétraitement

In [9]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical', seed=SEED),
    tf.keras.layers.RandomRotation(0.25, seed=SEED),
    tf.keras.layers.RandomZoom((-0.35, 0.1), seed=SEED),
    tf.keras.layers.RandomContrast(0.4, seed=SEED),
    tf.keras.layers.RandomBrightness(0.2, seed=SEED),
    tf.keras.layers.Lambda(
        lambda x: x + tf.random.normal(tf.shape(x), mean=0.0, stddev=8.0),
        name='gaussian_noise',
    ),
], name='data_augmentation')


def load_and_preprocess(path, label, augment=False):
    raw = tf.io.read_file(path)
    img = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    if augment:
        img = data_augmentation(img, training=True)
    return img, label


def preprocess_pv_tfds(image, label, augment=False):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    if augment:
        image = data_augmentation(image, training=True)
    return image, label

## 8. Construction des datasets TF

In [10]:
def dir_to_paths_labels(split_dir, label_to_idx: dict) -> tuple[list, list]:
    """Charge tous les chemins d'images d'un répertoire ImageFolder-style."""
    paths, labels = [], []
    exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    for cls_dir in sorted(split_dir.iterdir()):
        if not cls_dir.is_dir() or cls_dir.name not in label_to_idx:
            continue
        idx = label_to_idx[cls_dir.name]
        for p in cls_dir.iterdir():
            if p.is_file() and p.suffix.lower() in exts:
                paths.append(str(p))
                labels.append(idx)
    return paths, labels


# --- Dataset fusionné (Phase 1) ---
# On charge tout depuis la racine, puis on fait le split manuel
all_merged_paths, all_merged_labels = dir_to_paths_labels(DATASET_ROOT, label_to_idx)

rng = np.random.default_rng(SEED)
perm = rng.permutation(len(all_merged_paths))
merged_train_paths = [all_merged_paths[i] for i in perm]
merged_train_labels = [all_merged_labels[i] for i in perm]

n_val = int(0.15 * len(merged_train_paths))
merged_val_paths,   merged_val_labels   = merged_train_paths[:n_val],  merged_train_labels[:n_val]
merged_train_paths, merged_train_labels = merged_train_paths[n_val:],  merged_train_labels[n_val:]

merged_train_ds = (
    tf.data.Dataset.from_tensor_slices((merged_train_paths, merged_train_labels))
    .shuffle(8192, seed=SEED)
    .map(lambda p, l: load_and_preprocess(p, l, augment=True), num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
merged_val_ds = (
    tf.data.Dataset.from_tensor_slices((merged_val_paths, merged_val_labels))
    .map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# --- PlantDoc (Phase 2 fine-tuning) ---
ft_paths, ft_labels = [], []
for pd_cls, merged_idx in valid_pd_mapping.items():
    cls_dir = PLANTDOC_TRAIN_DIR / pd_cls
    if not cls_dir.exists():
        continue
    exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    for p in cls_dir.iterdir():
        if p.is_file() and p.suffix.lower() in exts:
            ft_paths.append(str(p))
            ft_labels.append(merged_idx)

rng = np.random.default_rng(SEED)
perm = rng.permutation(len(ft_paths))
ft_paths  = [ft_paths[i]  for i in perm]
ft_labels = [ft_labels[i] for i in perm]
n_val_ft  = int(0.2 * len(ft_paths))
ft_val_paths,   ft_val_labels   = ft_paths[:n_val_ft],  ft_labels[:n_val_ft]
ft_train_paths, ft_train_labels = ft_paths[n_val_ft:],  ft_labels[n_val_ft:]

ft_train_ds = (
    tf.data.Dataset.from_tensor_slices((ft_train_paths, ft_train_labels))
    .shuffle(2048, seed=SEED)
    .map(lambda p, l: load_and_preprocess(p, l, augment=True), num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
ft_val_ds = (
    tf.data.Dataset.from_tensor_slices((ft_val_paths, ft_val_labels))
    .map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# --- PlantVillage test (évaluation isolée) ---
pv_test_ds = (
    pv_test_raw
    .map(preprocess_pv_tfds, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print(f'Fusionné  — train : {len(merged_train_paths):,}  |  val : {len(merged_val_paths):,}')
print(f'PlantDoc  — train : {len(ft_train_paths):,}  |  val : {len(ft_val_paths):,}')
print(f'PV test   — {sum(1 for _ in pv_test_raw):,} images')

Fusionné  — train : 30,688  |  val : 5,415
PlantDoc  — train : 649  |  val : 162
PV test   — 8,145 images


## 9. Modèle : MobileNetV3Small

In [11]:
base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet',
    include_preprocessing=True,
)
base_model.trainable = False

inputs  = tf.keras.Input(shape=IMG_SIZE + (3,))
x       = base_model(inputs, training=False)
x       = tf.keras.layers.GlobalAveragePooling2D()(x)
x       = tf.keras.layers.Dropout(0.25)(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs, name='agroscan_merged_pv_wild')
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_P1),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

print(f'Paramètres totaux : {model.count_params():,}')
print(f'Backbone gelé — seule la tête ({num_classes} classes) est entraînable')

4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Paramètres totaux : 985,280
Backbone gelé — seule la tête (80 classes) est entraînable


## 10. Phase 1a — Entraînement tête (backbone gelé)

Entraînement de la tête de classification uniquement sur le dataset fusionné PlantVillage + PlantWild. Le backbone reste gelé pour conserver les features ImageNet.

In [ ]:
ckpt_p1a = str(MODEL_DIR / 'merged_wild_phase1a.keras')

callbacks_p1a = [
    tf.keras.callbacks.ModelCheckpoint(
        ckpt_p1a, monitor='val_accuracy', save_best_only=True,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=3, restore_best_weights=True,
    ),
]

history_p1a = model.fit(
    merged_train_ds,
    validation_data=merged_val_ds,
    epochs=EPOCHS_P1_HEAD,
    callbacks=callbacks_p1a,
)

pd.DataFrame(history_p1a.history).to_csv(REPORT_DIR / 'merged_wild_phase1a_history.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
pd.DataFrame(history_p1a.history)[['loss', 'val_loss']].plot(ax=axes[0], title='Phase 1a — Loss (fusionné, backbone gelé)')
pd.DataFrame(history_p1a.history)[['accuracy', 'val_accuracy']].plot(ax=axes[1], title='Phase 1a — Accuracy')
plt.tight_layout()
plt.show()

Epoch 1/5


I0000 00:00:1779714302.713436     130 service.cc:152] XLA service 0x799bf8002de0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779714302.713476     130 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1779714302.713481     130 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1779714304.220979     130 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1779714311.633426     130 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


564/959 ━━━━━━━━━━━━━━━━━━━━ 1:34 240ms/step - accuracy: 0.2942 - loss: 2.9828

## 11. Phase 1b — Dégel partiel du backbone (25 % des couches profondes)

On dégèle les 25 % de couches les plus profondes avec un LR réduit (×10) pour affiner les features sur les images terrain du dataset fusionné sans oubli catastrophique.

In [ ]:
backbone     = model.layers[1]  # MobileNetV3Small
n_layers     = len(backbone.layers)
freeze_until = int(UNFREEZE_RATIO * n_layers)  # 75% gelées → 25% dégelées

backbone.trainable = True
for layer in backbone.layers[:freeze_until]:
    layer.trainable = False

n_trainable      = sum(1 for l in backbone.layers if l.trainable)
trainable_params = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
total_params     = model.count_params()

print(f'Backbone : {freeze_until}/{n_layers} couches gelées, {n_trainable} dégelées')
print(f'Paramètres entraînables : {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.1f} %)')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_P1_UNFREEZE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

ckpt_p1b = str(MODEL_DIR / 'merged_wild_phase1b.keras')

callbacks_p1b = [
    tf.keras.callbacks.ModelCheckpoint(
        ckpt_p1b, monitor='val_accuracy', save_best_only=True,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=3, restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1,
    ),
]

history_p1b = model.fit(
    merged_train_ds,
    validation_data=merged_val_ds,
    epochs=EPOCHS_P1_FULL,
    callbacks=callbacks_p1b,
)

pd.DataFrame(history_p1b.history).to_csv(REPORT_DIR / 'merged_wild_phase1b_history.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
pd.DataFrame(history_p1b.history)[['loss', 'val_loss']].plot(ax=axes[0], title='Phase 1b — Loss (fusionné, dégel 25%)')
pd.DataFrame(history_p1b.history)[['accuracy', 'val_accuracy']].plot(ax=axes[1], title='Phase 1b — Accuracy')
plt.tight_layout()
plt.show()

## 12. Phase 2 — Fine-tuning léger sur PlantDoc

LR très faible (`5e-5`). Le dégel reste à 25 %. Objectif : adapter les features aux conditions réelles de PlantDoc (images de terrain, fond naturel) sans écraser ce qui a été appris sur le dataset fusionné.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_P2),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

ckpt_p2 = str(MODEL_DIR / 'merged_wild_phase2_plantdoc.keras')

callbacks_p2 = [
    tf.keras.callbacks.ModelCheckpoint(
        ckpt_p2, monitor='val_accuracy', save_best_only=True,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=3, restore_best_weights=True,
    ),
]

history_p2 = model.fit(
    ft_train_ds,
    validation_data=ft_val_ds,
    epochs=EPOCHS_P2,
    callbacks=callbacks_p2,
)

pd.DataFrame(history_p2.history).to_csv(REPORT_DIR / 'merged_wild_phase2_plantdoc_history.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
pd.DataFrame(history_p2.history)[['loss', 'val_loss']].plot(ax=axes[0], title='Phase 2 — Loss (fine-tuning PlantDoc)')
pd.DataFrame(history_p2.history)[['accuracy', 'val_accuracy']].plot(ax=axes[1], title='Phase 2 — Accuracy')
plt.tight_layout()
plt.show()

## 13. Évaluation sur PlantVillage (standalone)

Mesure la rétention de performance sur les images studio PlantVillage. Le modèle utilisé est celui **après fine-tuning PlantDoc** (Phase 2).

In [ ]:
def collect_predictions(model, ds):
    y_true, y_pred, y_conf = [], [], []
    for images, labels in ds:
        probs = model.predict(images, verbose=0)
        y_true.extend(labels.numpy().tolist())
        y_pred.extend(np.argmax(probs, axis=1).tolist())
        y_conf.extend(np.max(probs, axis=1).tolist())
    return np.array(y_true), np.array(y_pred), np.array(y_conf)


# Sur PlantVillage test, les labels tfds utilisent l'index PlantVillage.
# On les convertit vers l'index du dataset fusionné pour comparer avec les prédictions du modèle.
def remap_pv_labels(ds):
    """Convertit les labels PlantVillage (tfds) vers les index du dataset fusionné."""
    y_true_merged, images_list = [], []
    for images, labels in ds:
        for img, lbl in zip(images.numpy(), labels.numpy()):
            pv_cls = pv_class_names[int(lbl)]
            merged_idx = pv_to_merged_idx.get(pv_cls)
            if merged_idx is not None:
                images_list.append(img)
                y_true_merged.append(merged_idx)
    return images_list, y_true_merged

print('Collecte des prédictions sur PlantVillage test...')
images_pv, y_true_pv_remapped = remap_pv_labels(pv_test_ds)

y_pred_pv, y_conf_pv = [], []
batch_imgs = []
for i, img in enumerate(images_pv):
    batch_imgs.append(img)
    if len(batch_imgs) == BATCH_SIZE or i == len(images_pv) - 1:
        batch_tensor = tf.stack(batch_imgs)
        probs = model.predict(batch_tensor, verbose=0)
        y_pred_pv.extend(np.argmax(probs, axis=1).tolist())
        y_conf_pv.extend(np.max(probs, axis=1).tolist())
        batch_imgs = []

y_true_pv_remapped = np.array(y_true_pv_remapped)
y_pred_pv          = np.array(y_pred_pv)
y_conf_pv          = np.array(y_conf_pv)

pv_acc = accuracy_score(y_true_pv_remapped, y_pred_pv)
pv_f1  = f1_score(y_true_pv_remapped, y_pred_pv, average='macro', zero_division=0)

print(f'\n=== PlantVillage (standalone) ===')
print(f'Images évaluées : {len(y_true_pv_remapped):,}')
print(f'Accuracy        : {pv_acc:.4f}')
print(f'F1 macro        : {pv_f1:.4f}')
print(f'Confiance moy.  : {y_conf_pv.mean():.4f}')

In [ ]:
# On s'assure que le dossier d'export pointe vers l'espace de travail Kaggle
REPORT_DIR = Path('/kaggle/working/reports')
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Rapport par classe — PlantVillage
report_pv = classification_report(
    y_true_pv_remapped, y_pred_pv,
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)
report_pv_df = pd.DataFrame(report_pv).T
report_pv_df.to_csv(REPORT_DIR / 'merged_wild_pv_report.csv')

# Matrice de confusion
cm_pv = confusion_matrix(y_true_pv_remapped, y_pred_pv)
plt.figure(figsize=(14, 12))
sns.heatmap(cm_pv, cmap='Blues', cbar=False)
plt.title('Matrice de confusion — PlantVillage test (après fine-tuning PlantDoc)')
plt.xlabel('Classe prédite')
plt.ylabel('Classe réelle')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'merged_wild_pv_confusion_matrix.png', dpi=150)
plt.show()

## 14. Évaluation sur PlantDoc (standalone)

Mesure la performance sur le domaine cible : images de terrain avec fond naturel. C'est la métrique principale de cette approche.

In [ ]:
plantdoc_test_ds = tf.keras.utils.image_dataset_from_directory(
    PLANTDOC_TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)
pd_class_names_raw = plantdoc_test_ds.class_names

# Mapping index PlantDoc → index dataset fusionné
pd_raw_to_merged_idx = {
    pd_class_names_raw.index(pd_cls): merged_idx
    for pd_cls, merged_idx in valid_pd_mapping.items()
    if pd_cls in pd_class_names_raw
}

y_true_pd, y_pred_pd, y_conf_pd = [], [], []
for images, labels in plantdoc_test_ds:
    probs = model.predict(images, verbose=0)
    preds = np.argmax(probs, axis=1)
    confs = np.max(probs, axis=1)
    for pd_raw_idx, pred, conf in zip(labels.numpy(), preds, confs):
        pd_raw_idx = int(pd_raw_idx)
        if pd_raw_idx not in pd_raw_to_merged_idx:
            continue
        y_true_pd.append(pd_raw_to_merged_idx[pd_raw_idx])
        y_pred_pd.append(int(pred))
        y_conf_pd.append(float(conf))

y_true_pd = np.array(y_true_pd)
y_pred_pd = np.array(y_pred_pd)
y_conf_pd = np.array(y_conf_pd)

pd_acc = accuracy_score(y_true_pd, y_pred_pd)
pd_f1  = f1_score(y_true_pd, y_pred_pd, average='macro', zero_division=0)

print(f'\n=== PlantDoc (standalone) ===')
print(f'Images évaluées : {len(y_true_pd):,}  |  Classes : {len(valid_pd_mapping)}')
print(f'Accuracy        : {pd_acc:.4f}')
print(f'F1 macro        : {pd_f1:.4f}')
print(f'Confiance moy.  : {y_conf_pd.mean():.4f}')

In [ ]:
# Rapport par classe — PlantDoc
common_labels_pd = sorted(set(y_true_pd))
report_pd = classification_report(
    y_true_pd, y_pred_pd,
    labels=common_labels_pd,
    target_names=[class_names[i] for i in common_labels_pd],
    output_dict=True,
    zero_division=0,
)
report_pd_df = pd.DataFrame(report_pd).T
report_pd_df.to_csv(REPORT_DIR / 'merged_wild_pd_report.csv')

# Matrice de confusion
cm_pd = confusion_matrix(y_true_pd, y_pred_pd, labels=common_labels_pd)
plt.figure(figsize=(14, 12))
sns.heatmap(cm_pd, cmap='Oranges', cbar=False,
            xticklabels=[class_names[i] for i in common_labels_pd],
            yticklabels=[class_names[i] for i in common_labels_pd])
plt.title('Matrice de confusion — PlantDoc test')
plt.xlabel('Classe prédite')
plt.ylabel('Classe réelle')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.savefig(REPORT_DIR / 'merged_wild_pd_confusion_matrix.png', dpi=150)
plt.show()

print('\nTop 10 classes les moins bien reconnues (F1) :')
display(report_pd_df.drop(index=['accuracy', 'macro avg', 'weighted avg'], errors='ignore')
        .sort_values('f1-score')
        .head(10)[['precision', 'recall', 'f1-score', 'support']])

## 15. Analyse comparative — PlantVillage vs PlantDoc

Comparaison côte-à-côte des métriques sur les deux datasets pour évaluer le compromis studio ↔ terrain.

In [ ]:
# Résumé des métriques globales
metrics = {
    'Dataset':    ['PlantVillage', 'PlantDoc'],
    'Accuracy':   [pv_acc,         pd_acc],
    'F1 macro':   [pv_f1,          pd_f1],
    'Confiance':  [y_conf_pv.mean(), y_conf_pd.mean()],
}
df_compare = pd.DataFrame(metrics).set_index('Dataset')
df_compare.to_csv(REPORT_DIR / 'merged_wild_comparison.csv')

print('=== Comparaison PlantVillage vs PlantDoc ===')
print(df_compare.to_string(float_format='{:.4f}'.format))

# Barplot comparatif
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
colors = ['steelblue', 'darkorange']

for ax, col in zip(axes, ['Accuracy', 'F1 macro', 'Confiance']):
    df_compare[[col]].plot(kind='bar', ax=ax, color=colors, legend=False, rot=0)
    ax.set_title(col)
    ax.set_ylim(0, 1)
    ax.set_ylabel('')
    ax.set_xlabel('')
    for bar, val in zip(ax.patches, df_compare[col]):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

axes[0].set_title('Accuracy')
axes[1].set_title('F1 macro')
axes[2].set_title('Confiance moyenne')

handles = [plt.Rectangle((0,0), 1, 1, color=c) for c in colors]
fig.legend(handles, df_compare.index.tolist(), loc='upper right', fontsize=10)
fig.suptitle('Approche fusionné (PV+Wild) → fine-tuning PlantDoc\nComparaison PlantVillage vs PlantDoc', fontsize=12)
plt.tight_layout()
plt.savefig(REPORT_DIR / 'merged_wild_comparison_chart.png', dpi=150)
plt.show()

In [ ]:
# Comparaison per-class F1 sur les classes communes aux deux datasets
common_merged_ids = sorted(set(y_true_pv_remapped) & set(y_true_pd))

f1_pv_per_class = f1_score(
    y_true_pv_remapped, y_pred_pv,
    labels=common_merged_ids, average=None, zero_division=0,
)
f1_pd_per_class = f1_score(
    y_true_pd, y_pred_pd,
    labels=common_merged_ids, average=None, zero_division=0,
)

df_f1_compare = pd.DataFrame({
    'classe':      [class_names[i] for i in common_merged_ids],
    'F1 PlantVillage': f1_pv_per_class,
    'F1 PlantDoc':     f1_pd_per_class,
    'Delta (PDoc-PV)': f1_pd_per_class - f1_pv_per_class,
}).set_index('classe').sort_values('Delta (PDoc-PV)')

df_f1_compare.to_csv(REPORT_DIR / 'merged_wild_f1_per_class_comparison.csv')

fig, ax = plt.subplots(figsize=(14, max(6, len(common_merged_ids) * 0.35)))
x = np.arange(len(common_merged_ids))
width = 0.35
bars1 = ax.barh(x - width/2, df_f1_compare['F1 PlantVillage'], width, label='PlantVillage', color='steelblue', alpha=0.8)
bars2 = ax.barh(x + width/2, df_f1_compare['F1 PlantDoc'],     width, label='PlantDoc',     color='darkorange', alpha=0.8)
ax.set_yticks(x)
ax.set_yticklabels(df_f1_compare.index, fontsize=8)
ax.set_xlabel('F1-score')
ax.set_title('F1 par classe — PlantVillage vs PlantDoc (classes communes, triées par Delta)')
ax.legend()
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='seuil 0.5')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'merged_wild_f1_per_class.png', dpi=150)
plt.show()

print('\nClasses avec la plus grande amélioration sur PlantDoc :')
display(df_f1_compare.sort_values('Delta (PDoc-PV)', ascending=False).head(8))
print('\nClasses avec la plus grande dégradation sur PlantDoc :')
display(df_f1_compare.sort_values('Delta (PDoc-PV)').head(8))

## 16. Export TFLite & sauvegarde

In [ ]:
keras_path = MODEL_DIR / 'agroscan_merged_pv_wild.keras'
model.save(str(keras_path))

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_bytes = converter.convert()

tflite_path = MODEL_DIR / 'agroscan_merged_pv_wild_dynamic_quant.tflite'
tflite_path.write_bytes(tflite_bytes)

labels_path = MODEL_DIR / 'labels_merged_pv_wild.json'
labels_path.write_text(json.dumps(class_names, ensure_ascii=False, indent=2), encoding='utf-8')

print('Keras  :', keras_path,   f'({keras_path.stat().st_size / 1024 / 1024:.2f} Mo)')
print('TFLite :', tflite_path,  f'({tflite_path.stat().st_size / 1024 / 1024:.2f} Mo)')
print('Labels :', labels_path)

from IPython.display import display, FileLink
display(FileLink(str(tflite_path), result_html_prefix='TFLite → '))
display(FileLink(str(keras_path),  result_html_prefix='Keras  → '))
display(FileLink(str(labels_path), result_html_prefix='Labels → '))

## 17. Synthèse

### Pipeline

| Phase | Dataset | Backbone | LR | Epochs max |
|---|---|---|---|---|
| 1a — Head only | Fusionné (PV + Wild) | Gelé (100%) | 1e-3 | 5 |
| 1b — Partial unfreeze | Fusionné (PV + Wild) | 25% dégelé | 1e-4 | 8 |
| 2 — Fine-tuning | PlantDoc | 25% dégelé | 5e-5 | 10 |

### Avantages de l'approche fusionnée vs séquentielle

- **Un seul dataset** : plus besoin de télécharger PlantVillage et PlantWild séparément
- **Pas d'oubli inter-phases** : le modèle voit les deux distributions simultanément en Phase 1
- **Equilibre naturel** : si le dataset fusionné est équilibré, pas de dominance d'un domaine

### Points de vigilance

- Les noms de classes du dataset Kaggle peuvent différer légèrement de ceux de PlantVillage tfds → vérifier le mapping
- Si le dataset fusionné contient des classes non présentes dans PlantDoc, le fine-tuning Phase 2 peut dégrader ces classes
- Le nombre total d'images peut impacter le temps d'entraînement en Phase 1